In [ ]:
import torch
import torch.nn as nn


# Encoder
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers=1, drop_prob=0.3):
        super().__init__()

        self.word_embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim,
                            num_layers=n_layers,
                            dropout=drop_prob,
                            batch_first=True)
        self.dropout = nn.Dropout(drop_prob)

    def forward(self, x):
        # x: [batch_size, seq_len]

        x = self.word_embed(x)
        x = self.dropout(x)

        outputs, (hidden, cell) = self.lstm(x)

        # returning only final states (context)
        return hidden, cell


#  Decoder
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers=1, drop_prob=0.3):
        super().__init__()

        self.word_embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim,
                            num_layers=n_layers,
                            dropout=drop_prob,
                            batch_first=True)
        self.classifier = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(drop_prob)

    def forward(self, token, hidden, cell):
        # token: [batch_size]

        token = token.unsqueeze(1)  # make it [batch, 1]

        token = self.word_embed(token)
        token = self.dropout(token)

        output, (hidden, cell) = self.lstm(token, (hidden, cell))

        logits = self.classifier(output.squeeze(1))

        return logits, hidden, cell


# Example Run
if __name__ == "__main__":
    VOCAB = 9000
    EMBED = 256
    HIDDEN = 512
    LAYERS = 2

    encoder = Encoder(VOCAB, EMBED, HIDDEN, LAYERS)
    decoder = Decoder(VOCAB, EMBED, HIDDEN, LAYERS)

    print("\n--- Encoder ---\n", encoder)
    print("\n--- Decoder ---\n", decoder)

    total_params = sum(p.numel() for p in list(encoder.parameters()) + list(decoder.parameters()) if p.requires_grad)
    print("\nTotal Trainable Parameters:", total_params)


--- Encoder ---
 Encoder(
  (word_embed): Embedding(9000, 256)
  (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
)

--- Decoder ---
 Decoder(
  (word_embed): Embedding(9000, 256)
  (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.3)
  (classifier): Linear(in_features=512, out_features=9000, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)

Total Trainable Parameters: 16581416


In [ ]:
#
import torch
import torch.nn as nn
import torch.optim as optim
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Sample dataset (English → Spanish)
dataset = [
    ("hello", "hola"),
    ("good morning", "buenos dias"),
    ("how are you", "como estas"),
    ("thank you", "gracias"),
    ("i love you", "te amo"),
    ("see you later", "hasta luego"),
    ("what is your name", "como te llamas"),
    ("i am fine", "estoy bien"),
]

# Vocabulary builder
def make_vocab(sentences):
    vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
    for sent in sentences:
        for word in sent.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

src_vocab = make_vocab([x[0] for x in dataset])
tgt_vocab = make_vocab([x[1] for x in dataset])

tgt_idx2word = {i: w for w, i in tgt_vocab.items()}

# Encoder
class EncoderModel(nn.Module):
    def __init__(self, input_size, emb_dim, hid_dim, layers):
        super().__init__()
        self.embed = nn.Embedding(input_size, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, layers)

    def forward(self, x):
        emb = self.embed(x)
        _, (hidden, cell) = self.rnn(emb)
        return hidden, cell

# Decoder
class DecoderModel(nn.Module):
    def __init__(self, output_size, emb_dim, hid_dim, layers):
        super().__init__()
        self.embed = nn.Embedding(output_size, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, layers)
        self.fc = nn.Linear(hid_dim, output_size)

    def forward(self, token, hidden, cell):
        token = token.unsqueeze(0)
        emb = self.embed(token)
        out, (hidden, cell) = self.rnn(emb, (hidden, cell))
        pred = self.fc(out.squeeze(0))
        return pred, hidden, cell

# Seq2Seq Wrapper
class Seq2SeqModel(nn.Module):
    def __init__(self, enc, dec):
        super().__init__()
        self.encoder = enc
        self.decoder = dec

    def forward(self, src, trg, tf_ratio=0.6):
        trg_len, batch_size = trg.shape
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(trg_len, batch_size, vocab_size).to(device)

        hidden, cell = self.encoder(src)
        inp = trg[0]

        for t in range(1, trg_len):
            out, hidden, cell = self.decoder(inp, hidden, cell)
            outputs[t] = out

            teacher = random.random() < tf_ratio
            best = out.argmax(1)

            inp = trg[t] if teacher else best

        return outputs

# Hyperparameters
EMB = 128
HID = 256
LAYERS = 2

encoder = EncoderModel(len(src_vocab), EMB, HID, LAYERS)
decoder = DecoderModel(len(tgt_vocab), EMB, HID, LAYERS)

model = Seq2SeqModel(encoder, decoder).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

print("Model ready!")

/Users/jyotirajbanshi/week6/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Model ready!


In [ ]:
#training
# Convert sentence to tensor
def text_to_tensor(text, vocab):
    tokens = text.split()
    tokens = ["<sos>"] + tokens + ["<eos>"]
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
    return torch.tensor(ids, dtype=torch.long).to(device)

EPOCHS = 150

print("Training...\n")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for src_txt, tgt_txt in dataset:
        src = text_to_tensor(src_txt, src_vocab).unsqueeze(1)
        trg = text_to_tensor(tgt_txt, tgt_vocab).unsqueeze(1)

        optimizer.zero_grad()

        output = model(src, trg)

        vocab_size = output.shape[-1]
        output = output[1:].reshape(-1, vocab_size)
        trg = trg[1:].reshape(-1)

        loss = loss_fn(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataset):.4f}")

print("\nTraining Done!\n")


# Inference
def translate_text(sentence):
    model.eval()
    with torch.no_grad():
        src = text_to_tensor(sentence, src_vocab).unsqueeze(1)
        hidden, cell = model.encoder(src)

        token = torch.tensor([tgt_vocab["<sos>"]]).to(device)
        result = []

        for _ in range(15):
            out, hidden, cell = model.decoder(token, hidden, cell)
            pred = out.argmax(1).item()

            if pred == tgt_vocab["<eos>"]:
                break

            result.append(tgt_idx2word[pred])
            token = torch.tensor([pred]).to(device)

    return " ".join(result)


# Test
print("=== TEST OUTPUT ===\n")
tests = ["hello", "thank you", "i love you", "good morning"]

for t in tests:
    print(f"EN: {t}")
    print(f"ES: {translate_text(t)}")
    print("-" * 30)

Training...

Epoch 10, Loss: 0.2106
Epoch 20, Loss: 0.0088
Epoch 30, Loss: 0.0040
Epoch 40, Loss: 0.0024
Epoch 50, Loss: 0.0016
Epoch 60, Loss: 0.0012
Epoch 70, Loss: 0.0009
Epoch 80, Loss: 0.0008
Epoch 90, Loss: 0.0006
Epoch 100, Loss: 0.0005
Epoch 110, Loss: 0.0004
Epoch 120, Loss: 0.0004
Epoch 130, Loss: 0.0003
Epoch 140, Loss: 0.0003
Epoch 150, Loss: 0.0003

Training Done!

=== TEST OUTPUT ===

EN: hello
ES: hola
------------------------------
EN: thank you
ES: gracias
------------------------------
EN: i love you
ES: te amo
------------------------------
EN: good morning
ES: buenos dias
------------------------------


In [ ]:
print("English → Spanish Translator ")
print("Type 'exit' to stop\n")

while True:
    user_text = input("English: ").strip()

    if user_text.lower() in ["exit", "quit"]:
        print("Stopped.")
        break

    if not user_text:
        continue

    output = translate_text(user_text)

    print(f"Spanish: {output}")
